# Part (a): Time-series EDA

Timestamp quality and temporal dependence affect both preprocessing and forecast evaluation. This scaffold organizes the audit of the Beijing Multi-Site Air Quality data, hourly pollutant and weather readings from 12 stations, before any transformation is selected. The chapter selects one station, defines PM2.5 as the primary forecast target and PM10 as the secondary target, and describes the other variables that the preprocessing chapter uses for Granger tests.

Analysis cells are unfinished prompts and contain no measured results. For each section, add theory and any formula before the code, store and display measurements in named variables, and replace the interpretation prompt with conclusions supported by those outputs.


## Load the environment

The setup cell imports the shared paths and seed configuration and creates the experiment's random generator. Use the repository README for environment installation.


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from preprocessing import RAW_DATA_DIR, PROCESSED_DATA_DIR, OUTPUT_DIR, RANDOM_STATE

rng = np.random.default_rng(RANDOM_STATE)
sns.set_theme(style="whitegrid")
print(f"Random seed: {RANDOM_STATE}")
print(f"Raw data directory: {RAW_DATA_DIR}")

from statsmodels.graphics.tsaplots import plot_acf, plot_pacf


Random seed: 42
Raw data directory: D:\HCMUS\DataMining\preprocessing\data\raw


## Prepare data

The commented example below calls the shared helpers in `preprocessing.air_quality`. They download the UCI archive once, verify its SHA-256 checksum, and read the 12 station files without changing any values. Enable it after confirming access, keep the original `raw` frame unchanged, and record the source and acquisition steps here. Downloaded files stay outside version control.


In [3]:
# Uncomment to download and read the dataset (network access required on first run).
from preprocessing.air_quality import (
    AIR_QUALITY_SHA256, add_timestamp, audit_stations, download_air_quality, read_stations,
)
archive_path = download_air_quality(RAW_DATA_DIR)
raw = read_stations(archive_path)
stations = add_timestamp(raw)
print(f"Archive SHA-256: {AIR_QUALITY_SHA256}")
print(raw.shape, raw["station"].nunique())

Archive SHA-256: b04da438b2f331ac0ffd45aebdfec0d20d2367feb5f6948c4b1f7ce1191e33c4
(420768, 18) 12


In [6]:
stations

,No,year,month,day,hour,PM2.5,PM10,SO2,NO2,CO,O3,TEMP,PRES,DEWP,RAIN,wd,WSPM,station,timestamp
0,1,2013,3,1,0,4.0,4.0,4.0,7.0,300.0,77.0,-0.7,1023.0,-18.8,0.0,NNW,4.4,Aotizhongxin,2013-03-01 00:00:00
1,2,2013,3,1,1,8.0,8.0,4.0,7.0,300.0,77.0,-1.1,1023.2,-18.2,0.0,N,4.7,Aotizhongxin,2013-03-01 01:00:00
2,3,2013,3,1,2,7.0,7.0,5.0,10.0,300.0,73.0,-1.1,1023.5,-18.2,0.0,NNW,5.6,Aotizhongxin,2013-03-01 02:00:00
3,4,2013,3,1,3,6.0,6.0,11.0,11.0,300.0,72.0,-1.4,1024.5,-19.4,0.0,NW,3.1,Aotizhongxin,2013-03-01 03:00:00
4,5,2013,3,1,4,3.0,3.0,12.0,12.0,300.0,72.0,-2.0,1025.2,-19.5,0.0,N,2.0,Aotizhongxin,2013-03-01 04:00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
420763,35060,2017,2,28,19,11.0,32.0,3.0,24.0,400.0,72.0,12.5,1013.5,-16.2,0.0,NW,2.4,Wanshouxigong,2017-02-28 19:00:00
420764,35061,2017,2,28,20,13.0,32.0,3.0,41.0,500.0,50.0,11.6,1013.6,-15.1,0.0,WNW,0.9,Wanshouxigong,2017-02-28 20:00:00
420765,35062,2017,2,28,21,14.0,28.0,4.0,38.0,500.0,54.0,10.8,1014.2,-13.3,0.0,NW,1.1,Wanshouxigong,2017-02-28 21:00:00
420766,35063,2017,2,28,22,12.0,23.0,4.0,30.0,400.0,59.0,10.5,1014.4,-12.9,0.0,NNW,1.2,Wanshouxigong,2017-02-28 22:00:00


## Define variable roles

Each variable has a role that determines how much of the pipeline it receives. PM2.5 is the primary target: every preprocessing technique in the next chapter is applied to it and interpreted in full. PM10 is the secondary target: the same code runs on it with parameters fitted on its own training data, and the results appear in compact comparison tables. PM10 counts all particles up to 10 µm, so it includes the PM2.5 fraction by definition. A strong association between the two targets is expected rather than discovered.

The remaining numeric variables are SO2, NO2, CO, O3, TEMP, PRES, DEWP, RAIN, and WSPM. They enter the multivariate summaries in this chapter and the Granger tests in the preprocessing chapter, and they can serve as lagged forecast features. Wind direction `wd` is a categorical compass label, so tests that require numeric series cannot use it directly. Map its 16 labels to angles in 22.5° steps and combine them with wind speed into the components $u = -s\sin\theta$ and $v = -s\cos\theta$, where $s$ is WSPM in m/s and $\theta$ is the direction the wind blows from, in radians clockwise from north. Count missing direction labels and state how they are handled. RAIN is zero in most hours; report its share of zero values before deciding whether a Granger test on it is meaningful.

Record each variable's units from the UCI variable description, and store the target and covariate column lists in named variables for the audits below.


In [ ]:
# TODO: Implement the method. Store metrics in named variables and print them.


Interpretation TODO: list each variable's role and units, report the share of zero RAIN values and the count of missing direction labels, and justify any variable excluded from the Granger tests.


## Select the station

The 12 stations share the city's weather, but they differ in pollutant levels and in where their records are missing. Parameters fitted on one station, such as a Box-Cox $\lambda$, the chosen lags, or anomaly thresholds, come from that station's training data and do not transfer to another station without being refitted and checked. This chapter analyzes one station in depth and chooses it from data quality.

Compare all stations with `audit_stations`: record count, time span, file ordering, duplicate and missing timestamps, missing rate, and the longest run of consecutive hours without a value. Audit every column the analysis uses, both targets and all Granger covariates, with `audit_columns`, which runs `audit_stations` once per column. A station with complete PM2.5 records but long gaps in O3 or TEMP weakens the multivariate sections.

Lead with gap length rather than missing rate. Isolated missing hours are easy to reconstruct, but interpolation across a long gap invents values, and STL, ACF/PACF, Ljung-Box, and lag features all require a complete, regularly spaced series. The longest gap depends on a single event, so also compare `long_gap_missing_share`, the fraction of missing hours that lie in gaps longer than 24 hours. Use the missing rate to break ties.

State the rule and store the chosen station in a named variable. If the rule was written after viewing the audit table, say so in the interpretation. A choice based only on data-quality measures remains valid in that case: the measures do not depend on target values or model scores, so the choice does not consult outcomes in the test interval. Check whether a different reasonable rule chooses the same station. Also plot monthly missing rates for each station to see whether gaps cluster in particular periods. Clustered gaps bear on the MCAR assumption behind the artificial gaps in the preprocessing chapter.


In [5]:
# TODO: Implement the method. Store metrics in named variables and print them.
audit_stations(stations,"PM10")

,rows,start,end,sorted,duplicate_timestamps,missing_timestamps,target_missing,target_missing_rate,longest_target_gap_hours
station,,,,,,,,,
Aotizhongxin,35064,2013-03-01,2017-02-28 23:00:00,True,0,0,718,0.020477,343
Changping,35064,2013-03-01,2017-02-28 23:00:00,True,0,0,582,0.016598,144
Dingling,35064,2013-03-01,2017-02-28 23:00:00,True,0,0,656,0.018709,43
Dongsi,35064,2013-03-01,2017-02-28 23:00:00,True,0,0,553,0.015771,57
Guanyuan,35064,2013-03-01,2017-02-28 23:00:00,True,0,0,429,0.012235,44
Gucheng,35064,2013-03-01,2017-02-28 23:00:00,True,0,0,381,0.010866,25
Huairou,35064,2013-03-01,2017-02-28 23:00:00,True,0,0,777,0.022159,208
Nongzhanguan,35064,2013-03-01,2017-02-28 23:00:00,True,0,0,440,0.012548,55
Shunyi,35064,2013-03-01,2017-02-28 23:00:00,True,0,0,548,0.015629,38


Interpretation TODO: state the selection rule, the chosen station, and how it ranks on each audit measure for both targets and the covariates. Report whether an alternative rule gives the same choice. Describe whether missing values cluster in time or occur at the same hours across stations, and explain what the choice implies for applying conclusions to the other stations.


## Audit timestamps and define splits

Timestamp errors can create apparent gaps or repeated observations. For the selected station, record the source revision, target and covariate columns with units, frequency, time span, timezone convention, duplicates, and missing timestamps or values before sorting or regularizing the series.

Confirm at least two years of daily or hourly observations. Define one set of chronological training, validation, and test intervals for all variables and both targets, and store the boundaries in named variables so the preprocessing chapter can restate them. Use training data to select preprocessing and reserve the test interval for final evaluation.


In [ ]:
# TODO: Implement the method. Store metrics in named variables and print them.


Interpretation TODO: summarize the timestamp and value-quality findings, confirm the dataset constraints, and justify the split boundaries. Explain any deduplication or timezone policy before applying it.


## Inspect trend and seasonality

A time plot helps identify patterns that later transformations must account for. Plot the original training series of PM2.5 and PM10 with labeled axes and units, and describe possible daily, weekly, and annual patterns. Separate observed structure from assumptions about its causes.


In [ ]:
# TODO: Implement the method. Store metrics in named variables and print them.


Interpretation TODO: identify the visible trend, periodic patterns, and unusual intervals. State which observations support the seasonal periods considered in preprocessing and which require further checks.


## Describe relationships among variables

Same-hour correlations show which variables move together and help anticipate redundant forecast features. Compute Pearson and Spearman correlation matrices for the numeric variables, with the $u$ and $v$ wind components, on the training interval only. Spearman's coefficient measures monotonic association and is less sensitive than Pearson's to the right-skewed pollutant distributions. Plot both as labeled heatmaps and state how missing values are handled, for example pairwise deletion with the resulting pair counts.

Correlation describes simultaneous association only. The ACF/PACF section examines lagged dependence within each target, and the Granger tests in the preprocessing chapter examine lagged dependence between variables. Neither establishes causation.


In [ ]:
# TODO: Implement the method. Store metrics in named variables and print them.


Interpretation TODO: identify strongly associated pairs and differences between the Pearson and Spearman results. Discuss the expected PM2.5–PM10 association, shared dependence on weather, and which pairs could make forecast features redundant.


## Examine ACF and PACF

Autocorrelation describes dependence across lags; partial autocorrelation helps assess the contribution of a lag after accounting for intervening lags. Compute both for each target on training data and record an explicit maximum lag, for example $L=\lfloor\min(10\log_{10}n,n/5)\rfloor$, where $n$ is the observation count and $L$ is the largest lag in observations. Cap $L$ to the valid PACF range.

Label the lag units and explain the confidence-band convention. ACF and PACF need a regularly spaced series without missing values; state how native gaps are handled for these plots. Each target's training-only PACF informs its own forecast lag choice in the preprocessing chapter.


In [ ]:
# TODO: Implement the method. Store metrics in named variables and print them.


Interpretation TODO: for each target, describe the dominant lags and any seasonal repetitions, accounting for the plotted confidence bands. Explain how the lag units relate to the sampling frequency, which lags warrant testing as forecast features, and where the two targets differ.


## Compare rolling statistics

Changes in local mean and variation can indicate that a constant-distribution assumption needs closer examination. For each target, compare rolling mean and standard deviation over 7, 30, and 90 days. On a validated hourly grid, these windows contain 168, 720, and 2160 observations.

State the alignment, minimum observation count, and standard-deviation convention. These plots provide exploratory evidence; the preprocessing chapter contains the stationarity tests.


In [ ]:
# TODO: Implement the method. Store metrics in named variables and print them.


Interpretation TODO: compare local means and variation across the three windows. Describe boundary effects and periods where the statistics change, then explain which stationarity questions the plots leave unresolved.
